In [1]:
import json
import re
import core.utils as oa
from rapidfuzz import fuzz
import pandas as pd
import io
import os

import datetime
from typing import Iterable, List
from numpyencoder import NumpyEncoder
from pathlib import Path
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_core.documents.base import Document
from langchain_core.embeddings.embeddings import Embeddings
from langchain_core.runnables import chain
import core.vectorsearch as vecsearch

# root directory path
ROOT = Path(os.getcwd()).resolve().parents[0]

/home/janosch/anaconda3/envs/ma_orgelpredigt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/janosch/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading german: Package 'german' not found in index


In [2]:
date = datetime.datetime.now().strftime("%y-%m-%d_%H:%M")
date

'25-09-20_22:25'

In [3]:
cosine_cutoff = 0.3
model_name = "LaBSE"

In [4]:
similarity_table = {}
similarity_table['date'] = date
similarity_table['corpus'] = "all_sermons"
similarity_table['method'] = 'vector_search'
similarity_table['fuzziness'] = cosine_cutoff


In [5]:
sermons = ["E000036"]

In [8]:
query = "Ein feste Burg ist unser gott"
matches = retriever.invoke({"query": query})
matches

KeyError: 'page'

In [6]:
for sermon in sermons:
    for x in ["bibel", "lieder"]:
        model = SentenceTransformer(f'sentence-transformers/{model_name}')


        class EmbedSomething(Embeddings):
            def __init__(self,model) -> None:
                self.model = model

            def embed_documents(self,texts):
                t = self.model.encode(texts)
                return t.tolist()

            def embed_query(self, text: str) -> List[float]:
                t = self.model.encode(text)
                return t.tolist()

        emb = EmbedSomething(model)
        
        if x == "bibel":
            directory = str(ROOT / f"./chroma/chroma_db_bibel_{model_name}")
        else:
            directory = str(ROOT / f"./chroma/chroma_db_{model_name}")
        vectordb = Chroma(persist_directory = directory, embedding_function=emb)

        @chain
        def retriever(inputs: dict) -> tuple[Document]:
            query = inputs["query"]
            page = inputs.get("page")
            filter_criteria = {}
            if page:
                filter_criteria["source"] = str(page)
            if not filter_criteria:
                filter_criteria = None

            docs, scores = zip(
                *vectordb.similarity_search_with_score(
                    query,
                    k=1,
                    filter=filter_criteria
                )
            )
            for doc, score in zip(docs, scores):
                doc.metadata["score"] = score
            return docs

        hits = vecsearch.find_similarities(x, sermon, 80, retriever)
        hits = vecsearch.correct_inbetween_matches(hits, retriever)
        hits = vecsearch.add_inferred_matches(hits, sermon, retriever)
        
        filepath = filepath = ROOT / f"predictions/{sermon}_{x}_{similarity_table['method']}_{similarity_table['fuzziness']}_{similarity_table['date']}.csv"

        hits.to_csv(filepath, index=False)

        info = {}
        info["date"] = similarity_table['date']
        info["task"] = x
        info["method"] = similarity_table['method']
        info["fuzziness"] = similarity_table['fuzziness']
        info['file'] = str(filepath)

        # append metadata if file already exists,
        # otherwise create new file
        my_file = Path(ROOT / f"predictions/{sermon}_predictions.json")
        if my_file.is_file():
            with open(my_file, "r", encoding="utf-8") as f:
                predictions = json.load(f)
            predictions.append(info)
            with open(my_file, "w", encoding="utf-8") as f:
                json.dump(predictions, f, ensure_ascii=False)
        else:
            with open(ROOT / f"predictions/{sermon}_predictions.json", "x", encoding="utf-8") as f:
                json.dump([info], f, ensure_ascii=False)

/tmp/ipykernel_81830/386224010.py:24: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory = directory, embedding_function=emb)


starting with E000036


/home/janosch/Projects/Personal/ma-project/core/vectorsearch.py:147: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  guessed_hits = pd.concat([guessed_hits, new_matches])


starting with E000036
Couldn't split Fundstelle: 1039
Couldn't split Fundstelle: 1259
Couldn't split Fundstelle: 1280
Couldn't split Fundstelle: 1259
Couldn't split Fundstelle: 1280
Couldn't split Fundstelle: 1198
Couldn't split Fundstelle: 1280
Couldn't split Fundstelle: 1198
Couldn't split Fundstelle: 326
Couldn't split Fundstelle: 1198
Couldn't split Fundstelle: 326
Couldn't split Fundstelle: 496
Couldn't split Fundstelle: 326
Couldn't split Fundstelle: 496
Couldn't split Fundstelle: 460
Couldn't split Fundstelle: 496
Couldn't split Fundstelle: 460
Couldn't split Fundstelle: 819
Couldn't split Fundstelle: 1162
Couldn't split Fundstelle: 624
Couldn't split Fundstelle: 635
Couldn't split Fundstelle: 399
Couldn't split Fundstelle: 1145
Couldn't split Fundstelle: 1049
Couldn't split Fundstelle: 313
Couldn't split Fundstelle: 691
Couldn't split Fundstelle: 443
Couldn't split Fundstelle: 691
Couldn't split Fundstelle: 443
Couldn't split Fundstelle: 636
Couldn't split Fundstelle: 257
Could